# Phase 6 · S2 (B encoder–decoder + LoRA) generation tuning — same trained model, better decoding (no training)

The same fix that lifted S1 (validation ROUGE-1 0.461 → 0.521 with the same weights) applied to S2 (B encoder–decoder + LoRA): it was submitted with S1's original decoding (minimum length 8% of each chunk, no repetition penalty) — Codabench BERTScore 0.8033 / ROUGE-1 0.331 / ROUGE-L 0.204; validation 0.3265 / 0.2117 (length ×0.59, 39% repeated trigrams).

This notebook **does not train anything**. It loads the run's trained weights and prepared chunks from its output dataset (`P6_S2_B_EncDec_LoRA/checkpoints/best.pt`, `P6_S2_B_EncDec_LoRA/data_cache.pt`) and the run's own model and cached-generation code, adds a repetition penalty (Keskar et al., 2019), and compares decoding settings on 60 held-out validation documents:
- **minimum summary length per chunk:** 15%, 22% or 30% of the chunk's tokens (maximum 50%), × **repetition penalty** 1.2 or 1.35 — around S1's best (22% / 1.2);
- plus the run's original setting (8% / 1.0) for comparison.

The best (ROUGE-1 + ROUGE-L) is re-scored on all 150 validation documents and used for the 100 test summaries. Changing only generation settings is not a new experiment — it is an improved submission of the same run.

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one is enough) · internet not needed.
- Add Input → the run's output dataset **`Phase6_S2_Dataset`**, **`rohitbatra2004/arch-data-1` (Architecture B)** and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 45–90 minutes.

Outputs: `/kaggle/working/P6_S2_tuned/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — The source run's settings (needed to rebuild its model and read its data; training settings are unused here)
# and the decoding settings tuned in this notebook.
SOURCE_RUN = "P6_S2_B_EncDec_LoRA"                # the trained run whose checkpoint is re-decoded
RUN_NAME = "P6_S2_tuned"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Chunk-level summarization data.
CHUNK_TOKENS = 1000                 # document tokens per chunk (whole sentences); B's encoder reads ≤ 1,024 tokens
MAX_TARGET_TOKENS = 640             # partial-summary tokens per chunk used in training (longer targets are cut)
MAX_CHUNKS_PER_DOC = 40             # chunks kept per document (> 99.5% of documents complete)
VAL_DOCS = 150                      # held-out training documents for validation (never trained on)

# LoRA on every linear projection of the encoder and the decoder.
LORA_RANK = 8
LORA_ALPHA = 16.0

# Training.
EPOCHS = 2
BATCH_SEQS = 8                      # (chunk, summary) pairs per forward pass, split over both GPUs
GRAD_ACCUM = 2                      # effective batch = 16 pairs
LR = 3e-4                           # LoRA starts at B = 0
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.03
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000

# Generation (greedy, cached).
GEN_BATCH = 16
MIN_NEW_FRACTION = 0.08             # minimum summary tokens per chunk = 8% of the chunk's tokens …
MAX_NEW_FRACTION = 0.45             # … maximum = 45% of the chunk's tokens (+ 32), capped at MAX_TARGET_TOKENS
NO_REPEAT_NGRAM = 4
KEEP_FRACTIONS = (0.5, 0.7, 0.85, 1.0)   # share of each chunk summary's sentences kept (tuned on validation)
USE_AMP = DEVICE.type == "cuda"

# Decoding (the only thing tuned here).
MAX_NEW_FRACTION = 0.50                       # maximum summary tokens per chunk = 50% of the chunk (+ 32)
ORIGINAL_SETTING = (0.08, 1.0)                # what the run was submitted with
MIN_NEW_FRACTIONS = (0.15, 0.22, 0.30)        # minimum summary tokens per chunk
REPETITION_PENALTIES = (1.2, 1.35)            # penalty on tokens already generated for the chunk
TUNE_DOCS = 60
print("Run:", RUN_NAME, "| source:", SOURCE_RUN, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Load the source run's prepared data (chunks, held-out validation documents with references, test chunks) and its
# best trained weights; the data settings must equal this notebook's.
src_data = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/data_cache.pt", recursive=True))
src_ckpt = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/checkpoints/best.pt", recursive=True))
assert src_data and src_ckpt, f"Add the output dataset of {SOURCE_RUN} as an input."
data = torch.load(src_data[0], weights_only=False)
assert data["settings"]["chunk_tokens"] == CHUNK_TOKENS and data["settings"]["max_target_tokens"] == MAX_TARGET_TOKENS
best = torch.load(src_ckpt[0], map_location="cpu", weights_only=False)
print(f"{SOURCE_RUN}: {len(data['val'])} validation documents, {len(data['test'])} test documents | best epoch {best['epoch']} "
      f"(validation loss {best['val_loss']:.4f})")

In [ ]:
# Cell 4 — Tokenizer, special ids and the sentence / word patterns used by the source run (decoding and scoring).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]
print("Tokenizer ready")

In [ ]:
# Cell 5 — The complete Phase-1 Architecture B (encoder–decoder), same parameter names as Phase 1, loaded strictly.
# Differences from the Phase-1 code are only additions: padding masks, and cached incremental decoding (Cell 10).
b_paths = sorted(glob.glob("/kaggle/input/**/architecture_B/model_B_state_dict.pt", recursive=True))
assert b_paths, "Add the dataset rohitbatra2004/arch-data-1 as an input."
B_ROOT = Path(b_paths[0]).parent
B_METADATA = json.load(open(B_ROOT / "model_B_metadata.json"))
B_CONFIG = B_METADATA["model_config"]
assert B_CONFIG["pre_norm"] and B_CONFIG["tie_embeddings"], "Phase-1 B is pre-LN with tied embeddings"

def gelu(x):
    return F.gelu(x, approximate="tanh")

def heads(t, n):
    B, T, Cd = t.shape
    return t.view(B, T, n, Cd // n).transpose(1, 2)

class SelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        self.qkv = nn.Linear(cfg["d_model"], 3 * cfg["d_model"], bias=cfg["bias"])
        self.out_proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x, mask, cache=None):
        """mask: boolean, broadcastable to [B, 1, T, S]; cache: (k, v) of earlier decoder positions."""
        q, k, v = (heads(t, self.n_heads) for t in self.qkv(x).chunk(3, dim=-1))
        if cache is not None:
            k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        B, H, T, D = out.shape
        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * D)), (k, v)

class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        d, b = cfg["d_model"], cfg["bias"]
        self.q_proj, self.k_proj = nn.Linear(d, d, bias=b), nn.Linear(d, d, bias=b)
        self.v_proj, self.out_proj = nn.Linear(d, d, bias=b), nn.Linear(d, d, bias=b)

    def memory_kv(self, memory):
        return heads(self.k_proj(memory), self.n_heads), heads(self.v_proj(memory), self.n_heads)

    def forward(self, x, kv, src_mask):
        out = F.scaled_dot_product_attention(heads(self.q_proj(x), self.n_heads), kv[0], kv[1], attn_mask=src_mask)
        B, H, T, D = out.shape
        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * D))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.fc2(gelu(self.fc1(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(cfg["d_model"], bias=cfg["bias"]), nn.LayerNorm(cfg["d_model"], bias=cfg["bias"])
        self.self_attn, self.ffn = SelfAttention(cfg), FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.self_attn(self.norm1(x), mask)[0]
        return x + self.ffn(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, b = cfg["d_model"], cfg["bias"]
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d, bias=b), nn.LayerNorm(d, bias=b), nn.LayerNorm(d, bias=b)
        self.self_attn, self.cross_attn, self.ffn = SelfAttention(cfg), CrossAttention(cfg), FeedForward(cfg)

    def forward(self, x, self_mask, cross_kv, src_mask, cache=None):
        a, kv = self.self_attn(self.norm1(x), self_mask, cache)
        x = x + a
        x = x + self.cross_attn(self.norm2(x), cross_kv, src_mask)
        return x + self.ffn(self.norm3(x)), kv

class ArchitectureB(nn.Module):
    """Phase-1 encoder–decoder; dropout is not used here (frozen-model adaptation and generation)."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.d_model = d
        self.token_embedding = nn.Embedding(cfg["vocab_size"], d)
        self.encoder_position_embedding = nn.Embedding(cfg["src_context_length"], d)
        self.decoder_position_embedding = nn.Embedding(cfg["tgt_context_length"], d)
        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["n_encoder_layers"])])
        self.encoder_final_norm = nn.LayerNorm(d, bias=cfg["bias"])
        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg["n_decoder_layers"])])
        self.decoder_final_norm = nn.LayerNorm(d, bias=cfg["bias"])
        self.output_bias = nn.Parameter(torch.zeros(cfg["vocab_size"]))

    def embed(self, ids, position_table, start=0):
        positions = torch.arange(start, start + ids.size(1), device=ids.device)
        return self.token_embedding(ids) * math.sqrt(self.d_model) + position_table(positions)[None]

    def encode(self, src):
        src_mask = src.ne(PAD_ID)[:, None, None, :]
        x = self.embed(src, self.encoder_position_embedding)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return self.encoder_final_norm(x), src_mask

    def cross_kvs(self, memory):
        return [layer.cross_attn.memory_kv(memory) for layer in self.decoder_layers]

    def decode(self, dec, cross, src_mask, caches=None, start=0):
        T = dec.size(1)
        causal = torch.ones(T, start + T, dtype=torch.bool, device=dec.device).tril(start)[None, None]
        x = self.embed(dec, self.decoder_position_embedding, start)
        new = []
        for i, layer in enumerate(self.decoder_layers):
            x, kv = layer(x, causal, cross[i], src_mask, None if caches is None else caches[i])
            new.append(kv)
        return self.decoder_final_norm(x), new

    def logits(self, states):
        return F.linear(states, self.token_embedding.weight, self.output_bias)

saved = torch.load(b_paths[0], map_location="cpu", weights_only=True)
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved
check = ArchitectureB(B_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)                     # every Phase-1 tensor, including the decoder
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(B_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(B_CONFIG, indent=2))
print(f"Phase-1 Architecture B (encoder + decoder) verified from {B_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 6 — LoRA: y = W0·x + b + (α/r)·B(A(x)) on every linear projection of the encoder and decoder; only A and B train.
# B = 0 at start, so the adapted model initially equals Phase 1 (checked below on a short source/target pair).
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(x))

def make_lora_model():
    m = ArchitectureB(B_CONFIG)
    m.load_state_dict(PHASE1_STATE, strict=True)
    for p in m.parameters():
        p.requires_grad_(False)
    for layer in list(m.encoder_layers) + list(m.decoder_layers):
        modules = [layer.self_attn, layer.ffn] + ([layer.cross_attn] if hasattr(layer, "cross_attn") else [])
        for module in modules:
            for name, child in list(module.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(module, name, LoRALinear(child, LORA_RANK, LORA_ALPHA))
    return m

def full_forward(m, src, dec):
    memory, src_mask = m.encode(src)
    return m.logits(m.decode(dec, m.cross_kvs(memory), src_mask)[0])

probe_src, probe_dec = torch.randint(4, 1000, (2, 20)), torch.randint(4, 1000, (2, 7))
plain = ArchitectureB(B_CONFIG); plain.load_state_dict(PHASE1_STATE, strict=True); plain.eval()
adapted = make_lora_model().eval()
with torch.no_grad():
    assert torch.allclose(full_forward(plain, probe_src, probe_dec), full_forward(adapted, probe_src, probe_dec), atol=1e-4), \
        "LoRA with B = 0 must equal Phase 1"
lora_layers = [m for m in adapted.modules() if isinstance(m, LoRALinear)]
n_lora = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
print(f"LoRA on {len(lora_layers)} projections — {n_lora:,} trainable parameters ({100 * n_lora / n_backbone:.2f}%)")
del plain, adapted

In [ ]:
# Cell 7 — Sequence-to-sequence summariser: encoder reads the chunk, decoder predicts the summary (loss on summary + [EOS]).
class SummaryEncDec(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, src, dec, labels):
        """src [B, S], dec [B, T] (= [BOS] + summary), labels [B, T] (= summary + [EOS], -100 on padding)."""
        memory, src_mask = self.backbone.encode(src)
        states, _ = self.backbone.decode(dec, self.backbone.cross_kvs(memory), src_mask)
        keep = labels.ne(-100)
        logits = self.backbone.logits(states[keep]).float()
        return F.cross_entropy(logits, labels[keep], reduction="sum").unsqueeze(0), keep.sum().float().unsqueeze(0)

torch.manual_seed(SEED)
model = SummaryEncDec(make_lora_model()).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = list(TRAINABLE_NAMES)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
assert CHUNK_TOKENS + 2 <= B_CONFIG["src_context_length"] and MAX_TARGET_TOKENS + 1 <= B_CONFIG["tgt_context_length"]
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)} | "
      f"source ≤ {B_CONFIG['src_context_length']}, target ≤ {B_CONFIG['tgt_context_length']} tokens")

In [ ]:
# Cell 8 — Load the source run's trained weights into the rebuilt model (the parameter names must match exactly).
assert set(best["trainable"]) == set(SAVED_NAMES), "Checkpoint does not match this run's trainable parameters"
model.load_state_dict(best["trainable"], strict=False)
model.eval()
print(f"Loaded {len(SAVED_NAMES)} trained tensors from {src_ckpt[0]}")

In [ ]:
# Cell 9 — (the source run's generation code, plus an optional repetition penalty) Cached greedy generation for the encoder–decoder: the encoder runs once per chunk, cross-attention keys/values
# are computed once, decoder self-attention keys/values are cached; repeated 4-grams blocked, min / max lengths per chunk.
def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, sources, max_new, min_new, repetition_penalty=1.0):
    """sources: encoder token lists → generated summary token lists (without [EOS])."""
    bb.eval()
    B, S = len(sources), max(len(s) for s in sources)
    src = torch.full((B, S), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, s in enumerate(sources):
        src[b, :len(s)] = torch.tensor(s, device=DEVICE)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        memory, src_mask = bb.encode(src)
        cross = bb.cross_kvs(memory)
        token = torch.full((B, 1), BOS_ID, dtype=torch.long, device=DEVICE)
        caches, generated, done = None, [[] for _ in range(B)], [False] * B
        for step in range(max(max_new)):
            states, caches = bb.decode(token, cross, src_mask, caches, start=step)
            logits = bb.logits(states[:, -1]).float()
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            if repetition_penalty != 1.0:                         # CTRL-style penalty on tokens already generated
                for b, seq in enumerate(generated):
                    if seq:
                        idx = torch.tensor(sorted(set(seq)), device=DEVICE)
                        v = logits[b, idx]
                        logits[b, idx] = torch.where(v > 0, v / repetition_penalty, v * repetition_penalty)
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            nxt = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if nxt[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(nxt[b].item())
            if all(done):
                break
            token = nxt[:, None]
    return generated

def summarise(items, min_fraction, repetition_penalty, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + [EOS_ID] for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(min_fraction * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new, repetition_penalty)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 10 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 11 — Compare the decoding settings on TUNE_DOCS validation documents (keep all sentences of every chunk summary);
# the first row is the setting the run was submitted with.
tune_items = data["val"][:TUNE_DOCS]
tune_refs = [d["summary"] for d in tune_items]
grid = {}
for min_fraction, penalty in [ORIGINAL_SETTING] + [(m, p) for m in MIN_NEW_FRACTIONS for p in REPETITION_PENALTIES]:
    start_time = time.time()
    summaries = [join_summary(c) for c in summarise(tune_items, min_fraction, penalty)]
    grid[(min_fraction, penalty)] = score_summaries(tune_refs, summaries)
    print(f"  min length {min_fraction:.2f} · repetition penalty {penalty:.2f}: {describe(grid[(min_fraction, penalty)])} "
          f"| {(time.time() - start_time) / 60:.1f} min", flush=True)
BEST_MIN, BEST_PENALTY = max(grid, key=lambda k: grid[k]["rouge1"] + grid[k]["rougeL"])
print(f"Chosen: minimum length {BEST_MIN:.2f} of each chunk, repetition penalty {BEST_PENALTY:.2f} "
      f"(original {ORIGINAL_SETTING}: {grid[ORIGINAL_SETTING]['rouge1']:.4f} / {grid[ORIGINAL_SETTING]['rougeL']:.4f})")

In [ ]:
# Cell 12 — Re-score the chosen setting on all held-out validation documents (comparable with the run's own validation score).
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"], BEST_MIN, BEST_PENALTY)
val_result = score_summaries(refs, [join_summary(c) for c in val_chunk_summaries])
print(f"All {len(refs)} validation documents: {describe(val_result)} | {(time.time() - start_time) / 60:.1f} min")
print("\nExample (validation) — reference:\n", data["val"][0]["summary"][:700],
      "\n\nGenerated:\n", join_summary(val_chunk_summaries[0])[:700])

In [ ]:
# Cell 13 — Test summaries with the chosen setting, format checks, submission.zip and a run summary.
start_time = time.time()
test_chunk_summaries = summarise(data["test"], BEST_MIN, BEST_PENALTY)
prediction = [{"id": d["id"], "summary": join_summary(c)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert len({p["id"] for p in prediction}) == len(prediction) == len(data["test"])
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
json.dump({"source": SOURCE_RUN, "min_new_fraction": BEST_MIN, "repetition_penalty": BEST_PENALTY, "max_new_fraction": MAX_NEW_FRACTION,
           "grid": {f"{k[0]}_{k[1]}": v for k, v in grid.items()}, "validation_all": val_result,
           "test_words_median": int(np.median(words))}, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} (references ≈ 640) "
      f"| generation {(time.time() - start_time) / 60:.1f} min")